In [1]:
# Imports
import os
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import joblib
import random
import pickle
import numpy as np
from statistics import mean, stdev
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix, f1_score
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import RepeatedStratifiedKFold
from itertools import product
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    roc_auc_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from statistics import mean, stdev
import numpy as np
import torch, random, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import config
from preprocessing_utils import *

# --------------------------
# Set seeds for reproducibility
# --------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:

# =======================
# Simple Neural Network
# =======================
class SimpleMultimodalNet(nn.Module):
    def __init__(self, clin_dim, mrna_dim, mut_dim, hidden_dim=config.HIDDEN_DIM, dropout=config.DROPOUT, lr=config.LEARNING_RATE):
        super().__init__()

        # Separate encoders for each modality
        self.clinical_fc = nn.Sequential(
            nn.Linear(clin_dim, hidden_dim), 
            nn.ReLU(),
            nn.Dropout(dropout)
            )
        self.mrna_fc = nn.Sequential(
            nn.Linear(mrna_dim, hidden_dim), 
            nn.ReLU(),
            nn.Dropout(dropout)
            )
        self.mut_fc = nn.Sequential(
            nn.Linear(mut_dim, hidden_dim), 
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Fusion layer
        self.fusion_fc = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),  # Binary output
        )
        
    def forward(self, clin, mrna, mut):
        clin_emb = self.clinical_fc(clin)
        mrna_emb = self.mrna_fc(mrna)
        mut_emb = self.mut_fc(mut)
        
        # Concatenate embeddings
        fused = torch.cat([clin_emb, mrna_emb, mut_emb], dim=1)
        output = self.fusion_fc(fused)
        return output.squeeze()


In [3]:
def train_one_fold(train_loader, val_loader, model, optimizer, criterion, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    best_val_auroc, best_state, patience_counter = 0, None, 0

    for epoch in range(config.NUM_EPOCHS):
        model.train()
        for clin, mrna, mut, y in train_loader:
            clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(clin, mrna, mut)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for clin, mrna, mut, y in val_loader:
                clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
                probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
                all_probs.append(probs)
                all_labels.append(y.cpu().numpy())

        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)
        val_auroc = roc_auc_score(all_labels, all_probs)

        if val_auroc > best_val_auroc:
            best_val_auroc = val_auroc
            best_state = model.state_dict()
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= config.PATIENCE:
                break

    return best_state

def evaluate_with_threshold(model, loader, threshold):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for clin, mrna, mut, y in loader:
            clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
            probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y.cpu().numpy())

    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)
    preds = (all_probs >= threshold).astype(float)

    metrics = {
        'auroc': roc_auc_score(all_labels, all_probs),
        'auprc': average_precision_score(all_labels, all_probs),
        'precision': precision_score(all_labels, preds, zero_division=0),
        'recall': recall_score(all_labels, preds, zero_division=0),
        'f1': f1_score(all_labels, preds, zero_division=0)
    }
    return metrics


def run_kfold_gridsearch_with_preprocessing(
    clinical_df, mrna_df, mutation_df, labels,
    external_clinical_df, external_mrna_df, external_mutation_df, external_labels,
    param_grid,
    k=5,
    n_repeats=3,
    optimize_metric='f1'
):
    """
    Grid search with repeated stratified K-fold CV, fitting preprocessors and
    feature selectors (SelectFromModel) within each fold.
    """

    rskf = RepeatedStratifiedKFold(n_splits=k, n_repeats=n_repeats, random_state=config.SEED)
    indices = np.arange(len(labels))
    param_combinations = list(product(*param_grid.values()))
    best_hyperparams, best_score = None, -1
    results_summary = {}

    for params in param_combinations:
        param_dict = dict(zip(param_grid.keys(), params))
        print(f"\n===== Hyperparams: {param_dict} =====")

        fold_metrics = {'auroc': [], 'auprc': [], 'precision': [], 'recall': [], 'f1': []}
        external_metrics = {'auroc': [], 'auprc': [], 'precision': [], 'recall': [], 'f1': []}

        for fold, (train_idx, val_idx) in enumerate(rskf.split(indices, labels)):
            print(f"\n--- Fold {fold + 1}/{k} ---")

            clin_train, clin_val = clinical_df.iloc[train_idx], clinical_df.iloc[val_idx]
            mrna_train, mrna_val = mrna_df.iloc[train_idx], mrna_df.iloc[val_idx]
            mut_train, mut_val = mutation_df.iloc[train_idx], mutation_df.iloc[val_idx]
            y_train, y_val = labels.iloc[train_idx], labels.iloc[val_idx]

            clinical_prep = ClinicalPreprocessorWrapper(
                cols_to_remove=config.CLINICAL_COLS_TO_REMOVE,
                categorical_cols=config.CATEGORICAL_COLS,
                max_null_frac=config.CLINICAL_MAX_NULL_FRAC,
                uniform_thresh=config.CLINICAL_UNIFORM_THRESH,
            )
            mrna_prep = MrnaPreprocessorWrapper(
                max_null_frac=config.MAX_NULL_FRAC,
                uniform_thresh=config.UNIFORM_THRESHOLD,
                corr_thresh=config.CORRELATION_THRESHOLD,
                var_thresh=config.VARIANCE_THRESHOLD,
                re_run_pruning=config.RE_RUN_PRUNING,
                literature_genes=config.LITERATURE_GENES,
                correlated_genes_path=config.CORRELATED_GENES_PATH,
                use_stability_selection=False,
                random_state=config.SEED,
            )
            mutation_prep = MutationPreprocessorWrapper(
                max_null_frac=config.MUTATION_MAX_NULL_FRAC,
                uniform_thresh=config.MUTATION_UNIFORM_THRESH,
            )

            clinical_prep.fit(clin_train)
            mrna_prep.fit(mrna_train, y_train)
            mutation_prep.fit(mut_train)

            clin_train = clinical_prep.transform(clin_train)
            clin_val = clinical_prep.transform(clin_val)
            clin_ext = clinical_prep.transform(external_clinical_df.copy())

            mrna_train = mrna_prep.transform(mrna_train)
            mrna_val = mrna_prep.transform(mrna_val)
            mrna_ext = mrna_prep.transform(external_mrna_df.copy())

            mut_train = mutation_prep.transform(mut_train)
            mut_val = mutation_prep.transform(mut_val)
            mut_ext = mutation_prep.transform(external_mutation_df.copy())

            # === SelectFromModel ===
            base_estimator_mrna = param_dict.get('mrna_model', LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000))
            sfm_mrna = SelectFromModel(estimator=base_estimator_mrna,
                                       threshold=param_dict.get('mrna_threshold', 'median'))
            sfm_mrna.fit(mrna_train, y_train)
            mrna_train = sfm_mrna.transform(mrna_train)
            mrna_val = sfm_mrna.transform(mrna_val)
            mrna_ext = sfm_mrna.transform(mrna_ext)

            base_estimator_mut = param_dict.get('mut_model', LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000))
            sfm_mut = SelectFromModel(estimator=base_estimator_mut,
                                      threshold=param_dict.get('mut_threshold', 'median'))
            sfm_mut.fit(mut_train, y_train)
            mut_train = sfm_mut.transform(mut_train)
            mut_val = sfm_mut.transform(mut_val)
            mut_ext = sfm_mut.transform(mut_ext)

            # --- Build datasets and dataloaders ---
            def to_loader(c, m, mu, y, shuffle=False):
                import pandas as pd
                import numpy as np
                import torch
                from torch.utils.data import TensorDataset, DataLoader
            
                def check_numeric(df, name):
                    if isinstance(df, np.ndarray):
                        df = pd.DataFrame(df)
                    non_numeric_cols = []
                    for col in df.columns:
                        if not pd.api.types.is_numeric_dtype(df[col]):
                            non_numeric_cols.append(col)
                    if non_numeric_cols:
                        print(f"WARNING: {name} has non-numeric columns: {non_numeric_cols}")
                        # Print the first few rows of problematic columns
                        print(df[non_numeric_cols].head())
                    return df.to_numpy(dtype=np.float32)
                
                c = check_numeric(c, "Clinical")
                m = check_numeric(m, "mRNA")
                mu = check_numeric(mu, "Mutation")
            
                if isinstance(y, (pd.DataFrame, pd.Series)):
                    y = y.to_numpy(dtype=np.float32).reshape(-1, 1)
                else:
                    y = np.array(y, dtype=np.float32).reshape(-1, 1)
                y = y.squeeze() # converts from [x, 1] to [x] shape
            
                ds = TensorDataset(
                    torch.tensor(c),
                    torch.tensor(m),
                    torch.tensor(mu),
                    torch.tensor(y)
                )
                return DataLoader(ds, batch_size=config.BATCH_SIZE, shuffle=shuffle)

            train_loader = to_loader(clin_train, mrna_train, mut_train, y_train, shuffle=True)
            val_loader = to_loader(clin_val, mrna_val, mut_val, y_val)
            ext_loader = to_loader(clin_ext, mrna_ext, mut_ext, external_labels)

            model = SimpleMultimodalNet(clin_train.shape[1], mrna_train.shape[1], mut_train.shape[1],
                                        param_dict["hidden_dim"], param_dict["dropout"], param_dict["lr"]).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=param_dict.get('lr', 1e-3))
            pos_weight_value = torch.tensor((len(y_train) - y_train.sum()) / y_train.sum(), dtype=torch.float32).to(device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)

            best_state = train_one_fold(train_loader, val_loader, model, optimizer, criterion, config.SEED + fold)
            model.load_state_dict(best_state)

            # Threshold tuning, evaluation identical
            all_probs, all_labels = [], []
            with torch.no_grad():
                for clin, mrna, mut, y in val_loader:
                    clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
                    probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
                    all_probs.append(probs)
                    all_labels.append(y.cpu().numpy())
            all_probs = np.concatenate(all_probs)
            all_labels = np.concatenate(all_labels)

            thresholds = np.linspace(0.001, 0.9, 200)
            best_t, best_metric = 0.5, -1
            for t in thresholds:
                preds = (all_probs >= t).astype(float)
                score = f1_score(all_labels, preds, zero_division=0)
                if score > best_metric:
                    best_metric, best_t = score, t

            val_metrics = evaluate_with_threshold(model, val_loader, best_t)
            ext_metrics = evaluate_with_threshold(model, ext_loader, best_t)

            print(f"Val metrics: {val_metrics}")
            print(f"Ext metrics: {ext_metrics}")

            for k_ in fold_metrics.keys():
                fold_metrics[k_].append(val_metrics[k_])
                external_metrics[k_].append(ext_metrics[k_])

        results_summary[str(param_dict)] = {
            "internal": {k: (mean(v), stdev(v)) for k, v in fold_metrics.items()},
            "external": {k: (mean(v), stdev(v)) for k, v in external_metrics.items()}
        }

        mean_f1 = results_summary[str(param_dict)]["internal"]["f1"][0]
        if mean_f1 > best_score:
            best_score = mean_f1
            best_hyperparams = param_dict

    print(f"\n=== Best hyperparameters: {best_hyperparams} (mean F1 = {best_score:.4f}) ===")
    return results_summary, best_hyperparams



In [4]:
param_grid = {
    'dropout': [0],
    'hidden_dim': [32],
    'lr': [1e-3],
    'mrna_threshold': [0.01],
    'mut_threshold': ['median'],
}

X_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_train.joblib"))
y_train = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_train.joblib"))
X_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_val.joblib"))
y_val = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_val.joblib"))
X_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "X_test.joblib"))
y_test = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "y_test.joblib"))
clinical_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "clinical_cols.joblib"))
mrna_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mrna_cols.joblib"))
mutation_cols = joblib.load(os.path.join(config.SPLIT_DATA_DIR, "mutation_cols.joblib"))

# Split modalities
clinical_train = X_train[clinical_cols]
mrna_train = X_train[mrna_cols]
mutation_train = X_train[mutation_cols]

clinical_val = X_val[clinical_cols]
mrna_val = X_val[mrna_cols]
mutation_val = X_val[mutation_cols]

clinical_test = X_test[clinical_cols]
mrna_test = X_test[mrna_cols]
mutation_test = X_test[mutation_cols]

results_summary, best_hyperparams = run_kfold_gridsearch_with_preprocessing(
    clinical_train, mrna_train, mutation_train, y_train,
    clinical_test, mrna_test, mutation_test, y_test,
    param_grid,
    k=3,
    n_repeats=1,
    optimize_metric='f1'
)



===== Hyperparams: {'dropout': 0, 'hidden_dim': 32, 'lr': 0.001, 'mrna_threshold': 0.01, 'mut_threshold': 'median'} =====

--- Fold 1/3 ---
Val metrics: {'auroc': 0.6810661764705882, 'auprc': 0.4826969964087783, 'precision': 0.48484848484848486, 'recall': 0.9411764705882353, 'f1': 0.64}
Ext metrics: {'auroc': 0.7791666666666667, 'auprc': 0.603309177922181, 'precision': 0.5238095238095238, 'recall': 0.9166666666666666, 'f1': 0.6666666666666666}

--- Fold 2/3 ---
Val metrics: {'auroc': 0.6756272401433692, 'auprc': 0.5155921803496123, 'precision': 0.7142857142857143, 'recall': 0.5555555555555556, 'f1': 0.625}
Ext metrics: {'auroc': 0.5874999999999999, 'auprc': 0.49855993655658265, 'precision': 0.4166666666666667, 'recall': 0.4166666666666667, 'f1': 0.4166666666666667}

--- Fold 3/3 ---
Val metrics: {'auroc': 0.7777777777777777, 'auprc': 0.6570520262249586, 'precision': 0.6363636363636364, 'recall': 0.7777777777777778, 'f1': 0.7}
Ext metrics: {'auroc': 0.6375, 'auprc': 0.584731815119746, 

In [5]:
def print_results_summary(results_summary):
    """
    Nicely prints the results summary from the k-fold grid search.
    """
    for param_str, metrics in results_summary.items():
        print("="*60)
        print(f"Hyperparameters: {param_str}")
        print("-"*60)
        for phase in ["internal", "external"]:
            print(f"{phase.upper()} METRICS:")
            for metric, (mean_val, std_val) in metrics[phase].items():
                print(f"  {metric:10}: {mean_val:.4f} ± {std_val:.4f}")
        print("="*60 + "\n")

print_results_summary(results_summary)
print(best_hyperparams)
# Best hyperparameters: {'dropout': 0, 'hidden_dim': 64, 'lr': 0.001}

Hyperparameters: {'dropout': 0, 'hidden_dim': 32, 'lr': 0.001, 'mrna_threshold': 0.01, 'mut_threshold': 'median'}
------------------------------------------------------------
INTERNAL METRICS:
  auroc     : 0.7115 ± 0.0575
  auprc     : 0.5518 ± 0.0926
  precision : 0.6118 ± 0.1167
  recall    : 0.7582 ± 0.1936
  f1        : 0.6550 ± 0.0397
EXTERNAL METRICS:
  auroc     : 0.6681 ± 0.0994
  auprc     : 0.5622 ± 0.0559
  precision : 0.4468 ± 0.0672
  recall    : 0.6667 ± 0.2500
  f1        : 0.5278 ± 0.1273

{'dropout': 0, 'hidden_dim': 32, 'lr': 0.001, 'mrna_threshold': 0.01, 'mut_threshold': 'median'}


In [6]:
def retrain_and_evaluate_best_model(
    clinical_train, mrna_train, mutation_train, y_train,
    clinical_test, mrna_test, mutation_test, y_test,
    best_hyperparams
):
    """
    Retrains the model using the best hyperparameters on all training data,
    then evaluates on the test data.
    """

    print("\n===== Retraining model with best hyperparameters =====")
    print(best_hyperparams)

    # === Preprocessing ===
    clinical_prep = ClinicalPreprocessorWrapper(
        cols_to_remove=config.CLINICAL_COLS_TO_REMOVE,
        categorical_cols=config.CATEGORICAL_COLS,
        max_null_frac=config.CLINICAL_MAX_NULL_FRAC,
        uniform_thresh=config.CLINICAL_UNIFORM_THRESH,
    )
    mrna_prep = MrnaPreprocessorWrapper(
        max_null_frac=config.MAX_NULL_FRAC,
        uniform_thresh=config.UNIFORM_THRESHOLD,
        corr_thresh=config.CORRELATION_THRESHOLD,
        var_thresh=config.VARIANCE_THRESHOLD,
        re_run_pruning=config.RE_RUN_PRUNING,
        literature_genes=config.LITERATURE_GENES,
        correlated_genes_path=config.CORRELATED_GENES_PATH,
        use_stability_selection=False,
        random_state=config.SEED,
    )
    mutation_prep = MutationPreprocessorWrapper(
        max_null_frac=config.MUTATION_MAX_NULL_FRAC,
        uniform_thresh=config.MUTATION_UNIFORM_THRESH,
    )

    clinical_prep.fit(clinical_train)
    mrna_prep.fit(mrna_train, y_train)
    mutation_prep.fit(mutation_train)

    clin_train = clinical_prep.transform(clinical_train)
    clin_test = clinical_prep.transform(clinical_test)

    mrna_train = mrna_prep.transform(mrna_train)
    mrna_test = mrna_prep.transform(mrna_test)

    mut_train = mutation_prep.transform(mutation_train)
    mut_test = mutation_prep.transform(mutation_test)

    # === Feature selection ===
    base_estimator_mrna = best_hyperparams.get('mrna_model', LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000))
    sfm_mrna = SelectFromModel(estimator=base_estimator_mrna,
                               threshold=best_hyperparams.get('mrna_threshold', 'median'))
    mrna_test = mrna_test.reindex(columns=mrna_train.columns, fill_value=0)
    
    sfm_mrna.fit(mrna_train, y_train)
    mrna_train = sfm_mrna.transform(mrna_train)
    mrna_test = sfm_mrna.transform(mrna_test)

    base_estimator_mut = best_hyperparams.get('mut_model', LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000))
    sfm_mut = SelectFromModel(estimator=base_estimator_mut,
                              threshold=best_hyperparams.get('mut_threshold', 'median'))
    mut_test = mut_test.reindex(columns=mut_train.columns, fill_value=0)
    
    sfm_mut.fit(mut_train, y_train)
    mut_train = sfm_mut.transform(mut_train)
    mut_test = sfm_mut.transform(mut_test)

    # === Build dataloaders ===
    def to_loader(c, m, mu, y, shuffle=False):
        import pandas as pd, numpy as np, torch
        from torch.utils.data import TensorDataset, DataLoader

        def ensure_numeric(df):
            if isinstance(df, np.ndarray):
                return df
            return df.to_numpy(dtype=np.float32)
        
        c, m, mu = ensure_numeric(c), ensure_numeric(m), ensure_numeric(mu)
        y = np.array(y, dtype=np.float32).squeeze()
        ds = TensorDataset(
            torch.tensor(c, dtype=torch.float32),
            torch.tensor(m, dtype=torch.float32),
            torch.tensor(mu, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32)
        )
        return DataLoader(ds, batch_size=config.BATCH_SIZE, shuffle=shuffle)

    train_loader = to_loader(clin_train, mrna_train, mut_train, y_train, shuffle=True)
    test_loader = to_loader(clin_test, mrna_test, mut_test, y_test)

    # === Model ===
    model = SimpleMultimodalNet(
        clin_train.shape[1], mrna_train.shape[1], mut_train.shape[1],
        best_hyperparams["hidden_dim"], best_hyperparams["dropout"], best_hyperparams["lr"]
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=best_hyperparams.get('lr', 1e-3))
    pos_weight_value = torch.tensor((len(y_train) - y_train.sum()) / y_train.sum(), dtype=torch.float32).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)

    # === Training ===
    print("Training final model...")
    best_state = train_one_fold(train_loader, test_loader, model, optimizer, criterion, config.SEED)
    model.load_state_dict(best_state)

    # === Tune threshold on training set ===
    all_probs, all_labels = [], []
    with torch.no_grad():
        for clin, mrna, mut, y in train_loader:
            clin, mrna, mut, y = clin.to(device), mrna.to(device), mut.to(device), y.to(device)
            probs = torch.sigmoid(model(clin, mrna, mut)).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y.cpu().numpy())
    all_probs = np.concatenate(all_probs)
    all_labels = np.concatenate(all_labels)

    thresholds = np.linspace(0.001, 0.9, 200)
    best_t, best_metric = 0.5, -1
    for t in thresholds:
        preds = (all_probs >= t).astype(float)
        score = f1_score(all_labels, preds, zero_division=0)
        if score > best_metric:
            best_metric, best_t = score, t

    print(f"Best threshold from training: {best_t:.3f}")

    # === Final evaluation ===
    test_metrics = evaluate_with_threshold(model, test_loader, best_t)
    print(f"\nFinal test metrics: {test_metrics}")

    return model, test_metrics, best_t

best_hyperparams = {'dropout': 0, 'hidden_dim': 32, 'lr': 0.001, 'mrna_threshold': 0.01, 'mut_threshold': 'median'}

# Combine validation and test sets for each data type
clinical_combined = pd.concat([clinical_val, clinical_test], axis=0)
mrna_combined = pd.concat([mrna_val, mrna_test], axis=0)
mutation_combined = pd.concat([mutation_val, mutation_test], axis=0)

# Also combine the corresponding labels
y_combined = pd.concat([y_val, y_test], axis=0)

# Reset indices to avoid misalignment issues
clinical_combined = clinical_combined.reset_index(drop=True)
mrna_combined = mrna_combined.reset_index(drop=True)
mutation_combined = mutation_combined.reset_index(drop=True)
y_combined = y_combined.reset_index(drop=True)

model, test_metrics, best_t = retrain_and_evaluate_best_model(
    clinical_train, mrna_train, mutation_train, y_train,
    clinical_combined, mrna_combined, mutation_combined, y_combined,
    best_hyperparams
)

print("\n=== Final Test Results ===")
for metric, value in test_metrics.items():
    print(f"{metric}: {value:.4f}")

print(f"Optimal threshold: {best_t:.3f}")



===== Retraining model with best hyperparameters =====
{'dropout': 0, 'hidden_dim': 32, 'lr': 0.001, 'mrna_threshold': 0.01, 'mut_threshold': 'median'}
Training final model...
Best threshold from training: 0.001

Final test metrics: {'auroc': 0.7916666666666667, 'auprc': 0.7243456982246671, 'precision': 0.4583333333333333, 'recall': 0.9166666666666666, 'f1': 0.6111111111111112}

=== Final Test Results ===
auroc: 0.7917
auprc: 0.7243
precision: 0.4583
recall: 0.9167
f1: 0.6111
Optimal threshold: 0.001
